# 📊 Phase 2: Price Monitoring & Violation Audit Pipeline
### Production-Grade Dual-Table Architecture for MAP Auditing, Cross-Border Currency Parity & Infraction Governance

---

## 🎯 Executive Overview & Architecture
In omni-channel e-commerce distribution, brand equity and channel partner trust depend on rigorous, multi-tier compliance auditing. 
To serve both operational analytics and legal enforcement, this pipeline implements a clean **Dual-Table Architecture**:

```
                                  ┌─────────────────────────────┐
                                  │      Daily Seller Feeds     │
                                  │   (147,900 Observations)    │
                                  └──────────────┬──────────────┘
                                                 │
                                                 ▼
                                  ┌─────────────────────────────┐
                                  │   Relational Joins & Master  │
                                  │  Data Enrichment (MAP, LPP) │
                                  └──────────────┬──────────────┘
                                                 │
                                                 ▼
                                  ┌─────────────────────────────┐
                                  │  FX Normalization Engine    │
                                  │ (Cross-Border INR Currency) │
                                  └──────────────┬──────────────┘
                                                 │
                         ┌───────────────────────┴───────────────────────┐
                         ▼                                               ▼
         ┌───────────────────────────────┐               ┌───────────────────────────────┐
         │     Price Monitoring Table    │               │        Violation Table        │
         │     (Complete Audit Trail)    │               │     (Enforcement Ledger)      │
         │       • 147,900 Records       │               │        • 15,936 Records       │
         │       • Pure Data Dictionary  │               │        • Actionable Breaches  │
         │       • No Violation_Type col │               │        • Has Violation_Type,  │
         │                               │               │          Price_Difference &   │
         │                               │               │          Evidence Screenshot  │
         └───────────────────────────────┘               └───────────────────────────────┘
```

---

## 📐 Data Dictionaries & Target Schemas

### 1. `Price Monitoring Table` (14 Columns — Standard Audit Log)
Strictly adheres to the core Data Dictionary specification:
- `SKU`, `Violation_date`, `PL`, `Category`, `Sub_category`, `Region`, `Marketplace`, `seller_name`, `homologated_name`, `MAP_Price`, `LPP`, `Advertised_price`, `Season`, `Promotional_price`

### 2. `Violation Table` (18 Columns — Legal & Enforcement Ledger)
Contains only actionable infractions with quantified breach metrics and audit evidence:
- `SKU`, `Violation_date`, `PL`, `Category`, `Sub_category`, `Region`, `Marketplace`, `seller_name`, `homologated_name`, `MAP_Price`, `LPP`, `Advertised_price`, `Season`, `Promotion`, `Promotional_price`, `Screenshot`, `Violation_Type`, `Price_Difference`


### 1. Environment Setup & Configuration
We import standard data processing libraries, configure output formatting, and define the foreign exchange rate table (`FX_RATES`) for multi-regional currency normalization.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

# Project paths setup
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_CODE_DIR = PROJECT_ROOT / 'data_code'
DB_TABLES_DIR = DATA_CODE_DIR / 'database_tables'
INTERMEDIATE_DIR = DATA_CODE_DIR / 'intermediate'

# Business configuration constants
ACTIVE_SEASON = 'Q4'

# Foreign Exchange (FX) Conversion Rates to Base Currency (INR)
# Resolves cross-border comparison against INR-denominated MAP and LPP benchmarks
FX_RATES = {
    'IND': 1.0,      # Base currency (INR)
    'USA': 83.50,    # USD to INR
    'CAN': 61.50,    # CAD to INR
    'UK':  105.00,   # GBP to INR
    'UAE': 22.70     # AED to INR
}

print(f"Project Root:      {PROJECT_ROOT}")
print(f"Database Tables:  {DB_TABLES_DIR}")
print(f"Intermediate Dir: {INTERMEDIATE_DIR}")
print(f"Active Season:    {ACTIVE_SEASON}")
print(f"FX Rates Table:   {FX_RATES}")


Project Root:      D:\Projects\Map Data Analysis Project
Database Tables:  D:\Projects\Map Data Analysis Project\data_code\database_tables
Intermediate Dir: D:\Projects\Map Data Analysis Project\data_code\intermediate
Active Season:    Q4
FX Rates Table:   {'IND': 1.0, 'USA': 83.5, 'CAN': 61.5, 'UK': 105.0, 'UAE': 22.7}


### 2. Ingesting Database Tables and Promotion Master Data
We load:
1. **`seller_table.csv`**: Daily scraped seller pricing feeds (147,900 observations across 5 global regions).
2. **`seller_mapping_table.csv`**: Entity resolution mapping table resolving marketplace aliases to homologated parent companies.
3. **`category_mapping_table.csv`**: Taxonomy hierarchy linking SKUs to PL, Category, and Sub-category.
4. **`price_list_table.csv`**: Core price list with MAP and LPP benchmarks.
5. **`promotion_table.csv`**: Official seasonal promotions catalog.
6. **`promotional_skus.csv`**: White-list of SKUs authorized for promotional pricing.
7. **`promotion_condition.csv`**: Seasonal promotion business rules.


In [2]:
# Load relational tables
seller_df = pd.read_csv(DB_TABLES_DIR / 'seller_table.csv')
seller_map_df = pd.read_csv(DB_TABLES_DIR / 'seller_mapping_table.csv')
cat_map_df = pd.read_csv(DB_TABLES_DIR / 'category_mapping_table.csv')
price_list_df = pd.read_csv(DB_TABLES_DIR / 'price_list_table.csv')
prom_tab_df = pd.read_csv(DB_TABLES_DIR / 'promotion_table.csv')

# Load intermediate promotion governance files
promotional_skus_df = pd.read_csv(INTERMEDIATE_DIR / 'promotional_skus.csv')
prom_conditions_df = pd.read_csv(INTERMEDIATE_DIR / 'promotion_condition.csv')

# Convert authorized promotional SKUs into a fast lookup set
authorized_promotional_skus = set(promotional_skus_df['SKU'].dropna().unique())

print("=== Ingestion Summary ===")
print(f"- Seller Feed Rows:            {len(seller_df):,}")
print(f"- Seller Mappings:             {len(seller_map_df):,}")
print(f"- Category Mappings:           {len(cat_map_df):,}")
print(f"- Price List Entries:          {len(price_list_df):,}")
print(f"- Promotion Table Rows:        {len(prom_tab_df):,}")
print(f"- Authorized Promotional SKUs: {len(authorized_promotional_skus):,}")


=== Ingestion Summary ===
- Seller Feed Rows:            147,900
- Seller Mappings:             900
- Category Mappings:           1,546
- Price List Entries:          1,546
- Promotion Table Rows:        96
- Authorized Promotional SKUs: 100


### 3. Reviewing Seasonal Promotion Rules
Let's inspect the active promotional condition matrix across categories for **Q4**:

In [3]:
print("Promotional Rules Matrix:")
display(prom_conditions_df[['CATEGORY', 'Q4']])


Promotional Rules Matrix:


,CATEGORY,Q4
0,SUP,20% off ink
1,PH,$10 off laser jet
2,PC,15 % off


### 4. Data Integration & Cross-Border Currency Normalization
We join seller observations with master tables, retaining **all 147,900 records**.

We also link the active promotion terms from `promotion_table.csv` and calculate **`Advertised_price_INR`** using our `FX_RATES` matrix to ensure cross-border parity.

In [4]:
# Left join seller observations with homologated seller names
merged_df = seller_df.merge(
    seller_map_df,
    left_on='Seller_name',
    right_on='SELLER_NAME',
    how='left'
)

# Left join with category mapping hierarchy
merged_df = merged_df.merge(
    cat_map_df,
    on='SKU',
    how='left'
)

# Left join with price list benchmarks
merged_df = merged_df.merge(
    price_list_df[['SKU', 'MAP', 'LPP']],
    on='SKU',
    how='left'
)

# Left join with promotion table for Q4 promotion descriptions
q4_promo = prom_tab_df[prom_tab_df['SEASON'] == ACTIVE_SEASON][['SKU', 'PROMOTION']].drop_duplicates(subset=['SKU'])
merged_df = merged_df.merge(
    q4_promo,
    on='SKU',
    how='left'
)

# Currency Normalization Engine
merged_df['FX_Rate'] = merged_df['Region'].map(FX_RATES).fillna(1.0)
merged_df['Advertised_price_INR'] = np.round(merged_df['Advertised_price'] * merged_df['FX_Rate'], 2)

print(f"Total Master Observations: {len(merged_df):,}")
print(f"Rows with Catalog Price:   {merged_df['MAP'].notna().sum():,}")
print(f"Unlisted Catalog Rows:     {merged_df['MAP'].isna().sum():,}")
print("\nSample Normalized Prices across Regions:")
display(merged_df.groupby('Region')[['Advertised_price', 'FX_Rate', 'Advertised_price_INR', 'LPP']].head(1))


Total Master Observations: 147,900
Rows with Catalog Price:   143,550
Unlisted Catalog Rows:     4,350

Sample Normalized Prices across Regions:


,Advertised_price,FX_Rate,Advertised_price_INR,LPP
0,460.440,83.5,38446.74,30332.42
2,651.640,22.7,14792.23,12586.03
3,202.382,105.0,21250.11,17630.96
8,8630.000,1.0,8630.00,7537.45
10,134.240,61.5,8255.76,7537.45


### 5. Promotional Price Derivation Engine
According to the business charter and interview requirements:
1. **Authorization Gate**: Promotional prices apply **only** to SKUs listed in `promotional_skus.csv`.
2. **Category-Specific Q4 Rules**:
   - **PC**: $15\%$ discount applied directly to Advertised Price ($Advertised\_Price \times 0.85$).
   - **Supply (`SUP`)**: $20\%$ discount applied **only** if $Sub\_category == 'Ink'$.
   - **Print Hardware (`PH`)**: Fixed $10 off applied **only** if $Sub\_category == 'Laserjet'$. For the Indian market (`IND`), the $10 discount is converted to INR ($10 \times 83.50 = \text{₹}835$).
3. Non-eligible SKUs and categories retain `NaN` / null promotional pricing.


In [5]:
def calculate_promotional_price(row, promo_sku_set, fx_rates):
    """
    Computes the authorized promotional floor price for eligible SKUs based on Q4 rules in local currency.
    """
    sku = str(row['SKU']).strip()
    cat = str(row['CATEGORY']).strip()
    subcat = str(row['SUB_CATEGORY']).strip().lower()
    adv_price = row['Advertised_price']
    region = str(row['Region']).strip().upper()
    
    # Gate 1: Must be in authorized promotional SKUs whitelist
    if sku not in promo_sku_set:
        return np.nan
        
    # Rule 1: PC Category -> 15% discount
    if cat == 'PC':
        return round(adv_price * 0.85, 2)
        
    # Rule 2: Supply Category -> 20% discount strictly for Ink
    elif cat in ['SUP', 'Supply'] and subcat == 'ink':
        return round(adv_price * 0.80, 2)
        
    # Rule 3: Print Hardware -> $10 off strictly for Laserjet (converted to INR for IND)
    elif cat in ['PH', 'Print Hardware'] and 'laser' in subcat:
        usd_rate = fx_rates.get('USA', 83.50)
        discount_amount = 10.0 * usd_rate if region == 'IND' else 10.0
        return round(max(0.0, adv_price - discount_amount), 2)
        
    return np.nan

# Assign Season
merged_df['Season'] = ACTIVE_SEASON

# Apply promotional price calculation
merged_df['Promotional_price'] = merged_df.apply(
    lambda r: calculate_promotional_price(r, authorized_promotional_skus, FX_RATES),
    axis=1
)

promo_count = merged_df['Promotional_price'].notna().sum()
print(f"Calculated Promotional Prices: {promo_count:,} records ({promo_count/len(merged_df)*100:.2f}% of catalog)")
display(merged_df[merged_df['Promotional_price'].notna()][['SKU', 'CATEGORY', 'SUB_CATEGORY', 'Region', 'Advertised_price', 'Promotional_price']].head(5))


Calculated Promotional Prices: 7,335 records (4.96% of catalog)


,SKU,CATEGORY,SUB_CATEGORY,Region,Advertised_price,Promotional_price
12,81B19PA,PC,Laptop,UK,900.802,765.68
25,3YM22AA,SUP,Ink,USA,54.360,43.49
39,8L146PA,PC,Laptop,USA,3041.640,2585.39
70,381U0A,PH,Laserjet,USA,242.640,232.64
74,CN047AA,SUP,Ink,CAN,63.520,50.82


### 6. Compliance Auditing & Price Difference Derivation
Every observation is audited and tagged:
- **`Price Below LPP`**: Hard floor breach ($Advertised\_Price_{INR} < LPP$).
- **`Unauthorized Discount (Non-Promotional SKU)`**: Seller discounted a non-promotional SKU below MAP ($Advertised\_Price_{INR} < MAP$).
- **`Price_Difference`**: Exact financial undercut magnitude ($	ext{Threshold} - Advertised\_Price_{INR}$).


In [6]:
def audit_compliance_status(row, promo_sku_set):
    """
    Audits every record and classifies compliance or specific violation category.
    """
    if pd.isna(row['MAP']) or pd.isna(row['LPP']):
        return np.nan
        
    adv_inr = row['Advertised_price_INR']
    lpp = row['LPP']
    map_price = row['MAP']
    sku = str(row['SKU']).strip()
    is_promo = sku in promo_sku_set
    
    # Priority 1: Direct breach of Lowest Possible Price (LPP)
    if adv_inr < lpp:
        return 'Price Below LPP'
        
    # Priority 2: Non-promotional SKU carrying unauthorized discount below MAP
    elif not is_promo and adv_inr < map_price:
        return 'Unauthorized Discount (Non-Promotional SKU)'
        
    return np.nan

def calculate_price_difference(row):
    """
    Quantifies the price breach in INR for flagged violations.
    """
    v_type = row['Violation_Type']
    adv_inr = row['Advertised_price_INR']
    if v_type == 'Price Below LPP':
        return round(row['LPP'] - adv_inr, 2)
    elif v_type == 'Unauthorized Discount (Non-Promotional SKU)':
        return round(row['MAP'] - adv_inr, 2)
    return np.nan

# Tag violation status
merged_df['Violation_Type'] = merged_df.apply(
    lambda r: audit_compliance_status(r, authorized_promotional_skus),
    axis=1
)

# Compute Price_Difference
merged_df['Price_Difference'] = merged_df.apply(calculate_price_difference, axis=1)

print("=== Compliance Audit Summary ===")
print(merged_df['Violation_Type'].value_counts(dropna=False))


=== Compliance Audit Summary ===
Violation_Type
NaN                                            131964
Price Below LPP                                 14283
Unauthorized Discount (Non-Promotional SKU)      1653
Name: count, dtype: int64


### 7. Constructing the Dual-Table Output

We now shape both output tables cleanly:

1. **`Price Monitoring Table`** ($147,900$ rows):
   - Strict Data Dictionary columns (**14 columns**, no `Violation_Type`).
2. **`Violation Table`** ($15,936$ rows):
   - Contains only true violations with **`Violation_Type`**, **`Price_Difference`**, and **`Screenshot`** evidence (**18 columns**).


In [7]:
# Column renaming dictionary
schema_rename_map = {
    'Last time collected': 'Violation_date',
    'CATEGORY': 'Category',
    'SUB_CATEGORY': 'Sub_category',
    'Seller_name': 'seller_name',
    'HOMOLOGATED_NAME': 'homologated_name',
    'MAP': 'MAP_Price',
    'PROMOTION': 'Promotion'
}

master_processed_df = merged_df.rename(columns=schema_rename_map)

# 1. Price Monitoring Table (Exact Data Dictionary Schema - 14 Columns)
pm_columns = [
    'SKU',
    'Violation_date',
    'PL',
    'Category',
    'Sub_category',
    'Region',
    'Marketplace',
    'seller_name',
    'homologated_name',
    'MAP_Price',
    'LPP',
    'Advertised_price',
    'Season',
    'Promotional_price'
]
price_monitoring_table = master_processed_df[pm_columns].copy()

# 2. Violation Table (Enforcement Ledger - 18 Columns)
v_columns = [
    'SKU',
    'Violation_date',
    'PL',
    'Category',
    'Sub_category',
    'Region',
    'Marketplace',
    'seller_name',
    'homologated_name',
    'MAP_Price',
    'LPP',
    'Advertised_price',
    'Season',
    'Promotion',
    'Promotional_price',
    'Screenshot',
    'Violation_Type',
    'Price_Difference'
]
violation_table = master_processed_df[master_processed_df['Violation_Type'].notna()][v_columns].copy()

print(f"Price Monitoring Table Shape: {price_monitoring_table.shape}")
print(f"Violation Table Shape:        {violation_table.shape}")
print("\nPrice Monitoring Columns (14):", price_monitoring_table.columns.tolist())
print("\nViolation Table Columns (18):", violation_table.columns.tolist())


Price Monitoring Table Shape: (147900, 14)
Violation Table Shape:        (15936, 18)

Price Monitoring Columns (14): ['SKU', 'Violation_date', 'PL', 'Category', 'Sub_category', 'Region', 'Marketplace', 'seller_name', 'homologated_name', 'MAP_Price', 'LPP', 'Advertised_price', 'Season', 'Promotional_price']

Violation Table Columns (18): ['SKU', 'Violation_date', 'PL', 'Category', 'Sub_category', 'Region', 'Marketplace', 'seller_name', 'homologated_name', 'MAP_Price', 'LPP', 'Advertised_price', 'Season', 'Promotion', 'Promotional_price', 'Screenshot', 'Violation_Type', 'Price_Difference']


### 8. Compliance Intelligence & Offender Profiling
1. **Top Offending Sellers (Homologated)**: Identifies parent accounts generating the highest volume of infractions.
2. **Infractions by Category**: Distribution across PC, Supplies, and Print Hardware.
3. **Price Difference Severity**: Distribution of price undercut amounts.


In [8]:
print("=== 1. Top 10 Offending Resellers (Homologated) ===")
top_sellers = violation_table['homologated_name'].value_counts().head(10)
display(top_sellers.to_frame('Violation Count'))

print("\n=== 2. Violations by Category & Type ===")
cat_summary = violation_table.groupby(['Category', 'Violation_Type'])['SKU'].count().unstack(fill_value=0)
display(cat_summary)

print("\n=== 3. Price Difference Severity (INR Undercut Statistics) ===")
display(violation_table.groupby('Violation_Type')['Price_Difference'].describe().round(2))


=== 1. Top 10 Offending Resellers (Homologated) ===


,Violation Count
homologated_name,
Health Mart Systems,201
Sprouts Farmers Market,200
TJX Companies,199
Giant Eagle,195
Meijer,194
Wegmans Food Market,192
AVB Brandsource,192
True Value Co.,189
Dollar General,189



=== 2. Violations by Category & Type ===


Violation_Type,Price Below LPP,Unauthorized Discount (Non-Promotional SKU)
Category,,
PC,6241,543
PH,1603,108
SUP,6439,1002



=== 3. Price Difference Severity (INR Undercut Statistics) ===


,count,mean,std,min,25%,50%,75%,max
Violation_Type,,,,,,,,
Price Below LPP,14283.0,14720.52,32890.45,0.79,982.17,3828.44,14887.01,904483.3
Unauthorized Discount (Non-Promotional SKU),1653.0,1050.21,1700.31,0.56,121.83,436.61,1290.46,24908.0


### 9. Export to Database Tables
We export both tables to `data_code/database_tables/`:
1. `price_monitoring_table.csv` ($147,900$ rows)
2. `violation_table.csv` ($15,936$ rows)


In [9]:
# 1. Export Price Monitoring Table
pm_csv_path = DB_TABLES_DIR / 'price_monitoring_table.csv'
price_monitoring_table.to_csv(pm_csv_path, index=False)
print(f" Successfully exported Price Monitoring Table ({len(price_monitoring_table):,} rows):")
print(f"-> {pm_csv_path} ({pm_csv_path.stat().st_size / (1024 * 1024):.2f} MB)")

# 2. Export Violation Table
v_csv_path = DB_TABLES_DIR / 'violation_table.csv'
violation_table.to_csv(v_csv_path, index=False)
print(f"\n Successfully exported Violation Table ({len(violation_table):,} rows):")
print(f"-> {v_csv_path} ({v_csv_path.stat().st_size / (1024 * 1024):.2f} MB)")


 Successfully exported Price Monitoring Table (147,900 rows):
-> D:\Projects\Map Data Analysis Project\data_code\database_tables\price_monitoring_table.csv (15.90 MB)

 Successfully exported Violation Table (15,936 rows):
-> D:\Projects\Map Data Analysis Project\data_code\database_tables\violation_table.csv (2.18 MB)
